In [12]:
import os
import joblib
import pandas as pd
from trainingTools import getbest

In [3]:
ROUTE_DATA=r"C:\Users\Gabo\Downloads\models\models"
paths=os.listdir(ROUTE_DATA)
pathsPooling=[x for x in paths  if 'pooling' in x]
pathsBatch=[x for x in paths if 'seed' in x]
paths_level=[x for x in pathsBatch if 'level' in x]
paths_skill=[x for x in pathsBatch if 'skill' in x]
paths_subject=[x for x in pathsBatch if 'subject' in x]
paths_claridad=set(pathsBatch).difference(
    set(paths_level).union(paths_skill).union(paths_subject)
)

final_level=pd.concat([pd.read_csv(
    f"{ROUTE_DATA}/{x}", index_col=0
) for x in paths_level])

final_subject=pd.concat([pd.read_csv(
    f"{ROUTE_DATA}/{x}", index_col=0
) for x in paths_subject])

final_skill=pd.concat([pd.read_csv(
    f"{ROUTE_DATA}/{x}", index_col=0
) for x in paths_skill])

final_claridad=pd.concat([pd.read_csv(
    f"{ROUTE_DATA}/{x}", index_col=0
) for x in paths_claridad])

In [4]:
#parametros  para  obtener el pooling
groupcols=['pooling']
metrics= ['train_f1','val_f1','train_accuracy','val_accuracy']

pooling={}
for x in pathsPooling:
    db=pd.read_csv(f"{ROUTE_DATA}/{x}")
    cabezal=x.split('_')[1]
    best=getbest(db, groupcols,metrics)
    pooling[cabezal]=best['pooling']

print(pooling)

train_f1 :  0.8357930694832493
val_f1 :  0.7334743180949249
0.10231875138832436
{'claridad': 'mean', 'level': 'mean', 'skill': 'mean', 'subject': 'mean'}


In [5]:
groupcols=['num_hidden_layers',
       'hidden_dim', 'activation', 'normalization', 'dropout']

metrics= ['train_f1','val_f1','train_accuracy','val_accuracy']

a0=final_level[groupcols+metrics+['seed']]

a=a0[final_level['epoch']==final_level['best_epoch']]
b=a.groupby(groupcols)[metrics].agg('mean').reset_index()
print(a.shape)
print(b.shape)


(1296, 10)
(288, 9)


In [13]:
names=['level', 'skill','subject','claridad']
bases=[final_level,final_skill, final_subject, final_claridad]
info=dict(zip(names,bases))
params={}
for i,j in info.items():
    param=getbest(j,groupcols,metrics)
    print(i)
    print(param['train_f1'],param['val_f1'])
    pool=pooling[i]
    param['pooling']=pool
    params[i]=param
joblib.dump(params,'./finalCabezalParams.joblib')

level
0.9724864315861129 0.9300185014280814
train_f1 :  0.9324685606334997
val_f1 :  0.7133839932080378
0.2190845674254619
skill
0.9324685606334997 0.7133839932080378
subject
0.9981029466953156 0.9853864323226328
claridad
1.0 0.9985212792932189


['./finalCabezalParams.joblib']

In [14]:
params

{'level': {'num_hidden_layers': np.int64(1),
  'hidden_dim': np.int64(256),
  'activation': 'relu',
  'normalization': 'batchnorm',
  'dropout': np.float64(0.0),
  'train_f1': np.float64(0.9724864315861129),
  'val_f1': np.float64(0.9300185014280814),
  'train_accuracy': np.float64(0.973775433308214),
  'val_accuracy': np.float64(0.933778715424285),
  'no_overfiting': np.True_,
  'pooling': 'mean'},
 'skill': {'num_hidden_layers': np.int64(3),
  'hidden_dim': np.int64(512),
  'activation': 'relu',
  'normalization': 'batchnorm',
  'dropout': np.float64(0.0),
  'train_f1': np.float64(0.9324685606334997),
  'val_f1': np.float64(0.7133839932080378),
  'train_accuracy': np.float64(0.9116400685240285),
  'val_accuracy': np.float64(0.6973711882229233),
  'no_overfiting': np.False_,
  'pooling': 'mean'},
 'subject': {'num_hidden_layers': np.int64(3),
  'hidden_dim': np.int64(128),
  'activation': 'silu',
  'normalization': 'batchnorm',
  'dropout': np.float64(0.0),
  'train_f1': np.float64(0.

In [8]:
a0=final_skill[groupcols+metrics+['seed']]

a=a0[final_skill['epoch']==final_skill['best_epoch']]
b=a.groupby(groupcols)[metrics].agg('mean').reset_index()
print(a.shape)
print(b.shape)


(1296, 10)
(288, 9)
